# Trademark Similarity Pipeline (Kaggle)

This notebook builds the evaluation prompt and runs inference via a remote LLM using the Hugging Face Inference API (no local model download needed).

Instructions:
- Enable Internet for the Kaggle notebook runtime.
- Add a Kaggle Secret named `HF_API_TOKEN` containing your Hugging Face token.
- Upload this project as a Kaggle Dataset (it should contain `data/fewshot_cases.json` and `data/nice_chunks.json`).
- Update `DATASET_ROOT` below to your dataset input path (e.g., `/kaggle/input/your-dataset-slug`).



In [ ]:
import os
os.environ["HF_API_TOKEN"] = "..."



In [2]:
import os
import json
import re
import textwrap
import requests
from typing import List, Dict, Optional

# Config
FEWSHOT_PATH = "/kaggle/input/nice-chunk/fewshot_cases.json"
NICE_PATH = "/kaggle/input/nice-chunk/nice_chunks.json"

HF_API_TOKEN = os.getenv("HF_API_TOKEN", "")  # set via Kaggle Secrets
HF_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"  # remote hosted model; change as you like

assert os.path.exists(FEWSHOT_PATH), f"Missing {FEWSHOT_PATH}"
assert os.path.exists(NICE_PATH), f"Missing {NICE_PATH}"
assert HF_API_TOKEN, "Please set HF_API_TOKEN in Kaggle Secrets"

with open(FEWSHOT_PATH, "r", encoding="utf-8") as f:
    fewshot_cases: List[Dict] = json.load(f)

with open(NICE_PATH, "r", encoding="utf-8") as f:
    nice_chunks: List[Dict] = json.load(f)


def format_fewshot(example: Dict) -> str:
    case_id = example.get("case_id", "UNKNOWN")
    input_data = example.get("input", {})
    reasoning = example.get("reasoning", {})
    output = example.get("output", {})

    reasoning_lines = [f"- {k.capitalize()}: {v}" for k, v in reasoning.items()]

    return (
        f"### Example {case_id}\n"
        f"**Input:**\n"
        f"- Product 1: {input_data.get('product_1')}\n"
        f"- Product 2: {input_data.get('product_2')}\n"
        f"- Class Info: {json.dumps(input_data.get('class_info', {}), ensure_ascii=False, indent=2)}\n\n"
        f"**Reasoning:**\n" + "\n".join(reasoning_lines) + "\n\n"
        f"**Output:**\n"
        f"- Nature Score: {output.get('nature_score')}\n"
        f"- Purpose Score: {output.get('purpose_score')}\n"
        f"- Overall Similarity: {output.get('overall_similarity')}\n"
    )


def build_prompt(fewshot_examples: List[Dict],
                 product_1: str,
                 product_2: str,
                 retrieved_contexts: List[str],
                 max_fewshot: Optional[int] = 2) -> str:
    # Instruction
    instruction = (
        "You are an expert in trademark law and NICE classification.\n"
        "Your task is to evaluate the similarity between two products "
        "according to the official NICE classification and the 8 guideline factors: Nature, Intended Purpose, method of use, complementarity, competition, distribution channels, relevant public,the usual origin of the goods/services\n\n"
        "Scoring system (for each factor and overall):\n"
        "0 = Not similar, 1 = Slightly similar, 2 = Somewhat similar, "
        "3 = Similar, 4 = Highly similar.\n\n"
    )

    # Few-shot
    examples = fewshot_examples[:max_fewshot] if isinstance(max_fewshot, int) and max_fewshot > 0 else fewshot_examples
    fewshot_text = "\n\n".join([format_fewshot(ex) for ex in examples])

    # Contexts
    context_text = (
        "### Reference Context (NICE classification & guidelines):\n"
        + "\n".join(retrieved_contexts)
        + "\n\n"
    )

    # Query
    query_text = (
        "### New Case\n"
        f"- Product 1: {product_1}\n"
        f"- Product 2: {product_2}\n\n"
        "Output:\n"
        "- Nature Score: [0–4]\n"
        "- Purpose Score: [0–4]\n"
        "- Overall Similarity: [0–4]\n"
    )

    return instruction + fewshot_text + "\n\n" + context_text + query_text


def retrieve_contexts(product_1: str, product_2: str, top_k: int = 3) -> List[str]:
    terms = set([t for t in re.findall(r"[A-Za-z]+", (product_1 + " " + product_2).lower()) if len(t) > 3])
    scored = []
    for entry in nice_chunks:
        heading = entry.get("heading", "")
        note = entry.get("explanatory_note", "")
        items = entry.get("items", [])
        class_no = entry.get("class_number", "?")
        blob = (heading + "\n" + note + "\n" + "\n".join([it.get("Goods and Service", "") for it in items])).lower()
        score = sum(blob.count(term) for term in terms)
        if score > 0:
            matched_items = [it.get("Goods and Service", "") for it in items if any(term in it.get("Goods and Service", "").lower() for term in terms)]
            snippet_items = "; ".join(matched_items[:3])
            context = f"Class {class_no}: {heading}"
            if snippet_items:
                context += f"\nExamples: {snippet_items}"
            scored.append((score, context))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [ctx for _, ctx in scored[:top_k]]

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

def hf_inference(
    model_id: str = "Qwen/Qwen3-4B-Instruct-2507",
    prompt: str = "",
    temperature: float = 0.7,
    top_p: float = 0.9,
    max_new_tokens: int = 4096,
) -> str:
    # Load model + tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    # Encode input
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate
    outputs = model.generate(
        **inputs,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        max_new_tokens=max_new_tokens,
    )

    # Decode result
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result.strip()


In [4]:
# Test thử với 1 case trước khi loop
p1 = "Chemicals for industrial use"
p2 = "chemical additives for detergents"

contexts = retrieve_contexts(p1, p2, top_k=3)
prompt = build_prompt(fewshot_cases, p1, p2, contexts, max_fewshot=2)

print("=== PROMPT (full) ===")
print(prompt)

print("\n=== PROMPT (preview) ===")
import textwrap
print(textwrap.shorten(prompt, width=800, placeholder="..."))


=== PROMPT (full) ===
You are an expert in trademark law and NICE classification.
Your task is to evaluate the similarity between two products according to the official NICE classification and the 8 guideline factors: Nature, Intended Purpose, method of use, complementarity, competition, distribution channels, relevant public,the usual origin of the goods/services

Scoring system (for each factor and overall):
0 = Not similar, 1 = Slightly similar, 2 = Somewhat similar, 3 = Similar, 4 = Highly similar.

### Example USECASE_1
**Input:**
- Product 1: Make-up preparations (Class 3, Code 030033)
- Product 2: Tissues of paper for removing make-up (Class 16, Code 160294)
- Class Info: {
  "class_3": "Non-medicated toiletry preparations and cleaning products.",
  "class_16": "Mainly paper, cardboard and goods made from these materials."
}

**Reasoning:**
- Nature: Make-up preparations are chemical/beauty compositions, while tissues are paper products. Their physical nature is different.
- Int

In [5]:
import textwrap
import csv
import re

# File log
LOG_FILE = "results_log_1.csv"

# Hàm parse số điểm từ output
def parse_scores(output: str):
    scores = {"nature": None, "purpose": None, "overall": None}
    if not output:
        return scores

    patterns = {
        "nature": r"Nature Score:\s*(\d)",
        "purpose": r"Purpose Score:\s*(\d)",
        "overall": r"Overall (?:Similarity|Score):\s*(\d)"
    }

    for key, pat in patterns.items():
        m = re.search(pat, output, re.IGNORECASE)
        if m:
            scores[key] = int(m.group(1))
    return scores


# Các case muốn test
test_cases = [
    (
        "Chemicals for industrial use",
        "chemical additives for detergents"
    ),
    (
        "chemical products used in the manufacture of plastics and in the photocopying industry",
        "plastics in the form of granules, powders, masses, resins, gels, emulsions, dispersions, pastes, flakes, chips"
    ),
    (
        "Paints",
        "construction materials"
    )
]

# Mở file CSV và chạy test
with open(LOG_FILE, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["case_id", "p1", "p2", "nature_score", "purpose_score", "overall_score"])  # header

    for idx, (p1, p2) in enumerate(test_cases, start=1):
        print(f"\n=== CASE {idx} ===")
        contexts = retrieve_contexts(p1, p2, top_k=3)
        prompt = build_prompt(fewshot_cases, p1, p2, contexts, max_fewshot=2)

        # chỉ preview prompt (nếu cần debug), không ghi vào log
        print("=== PROMPT (preview) ===")
        print(textwrap.shorten(prompt, width=800, placeholder="..."))

        print("\n=== REMOTE INFERENCE ===")
        try:
            output = hf_inference(HF_MODEL_ID, prompt)
            print(output)

            scores = parse_scores(output)
            writer.writerow([
                idx, p1, p2,
                scores["nature"], scores["purpose"], scores["overall"]
            ])
        except Exception as e:
            err_msg = f"ERROR: {e}"
            print("Remote inference failed:", str(e))
            writer.writerow([idx, p1, p2, err_msg, "", ""])

print(f"\n✅ Log saved to {LOG_FILE}")



=== CASE 1 ===
=== PROMPT (preview) ===
You are an expert in trademark law and NICE classification. Your task is to evaluate the similarity between two products according to the official NICE classification and the 8 guideline factors: Nature, Intended Purpose, method of use, complementarity, competition, distribution channels, relevant public,the usual origin of the goods/services Scoring system (for each factor and overall): 0 = Not similar, 1 = Slightly similar, 2 = Somewhat similar, 3 = Similar, 4 = Highly similar. ### Example USECASE_1 **Input:** - Product 1: Make-up preparations (Class 3, Code 030033) - Product 2: Tissues of paper for removing make-up (Class 16, Code 160294) - Class Info: { "class_3": "Non-medicated toiletry preparations and cleaning products.", "class_16": "Mainly paper, cardboard and goods made from...

=== REMOTE INFERENCE ===


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

You are an expert in trademark law and NICE classification.
Your task is to evaluate the similarity between two products according to the official NICE classification and the 8 guideline factors: Nature, Intended Purpose, method of use, complementarity, competition, distribution channels, relevant public,the usual origin of the goods/services

Scoring system (for each factor and overall):
0 = Not similar, 1 = Slightly similar, 2 = Somewhat similar, 3 = Similar, 4 = Highly similar.

### Example USECASE_1
**Input:**
- Product 1: Make-up preparations (Class 3, Code 030033)
- Product 2: Tissues of paper for removing make-up (Class 16, Code 160294)
- Class Info: {
  "class_3": "Non-medicated toiletry preparations and cleaning products.",
  "class_16": "Mainly paper, cardboard and goods made from these materials."
}

**Reasoning:**
- Nature: Make-up preparations are chemical/beauty compositions, while tissues are paper products. Their physical nature is different.
- Intended_purpose: Both ar

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

You are an expert in trademark law and NICE classification.
Your task is to evaluate the similarity between two products according to the official NICE classification and the 8 guideline factors: Nature, Intended Purpose, method of use, complementarity, competition, distribution channels, relevant public,the usual origin of the goods/services

Scoring system (for each factor and overall):
0 = Not similar, 1 = Slightly similar, 2 = Somewhat similar, 3 = Similar, 4 = Highly similar.

### Example USECASE_1
**Input:**
- Product 1: Make-up preparations (Class 3, Code 030033)
- Product 2: Tissues of paper for removing make-up (Class 16, Code 160294)
- Class Info: {
  "class_3": "Non-medicated toiletry preparations and cleaning products.",
  "class_16": "Mainly paper, cardboard and goods made from these materials."
}

**Reasoning:**
- Nature: Make-up preparations are chemical/beauty compositions, while tissues are paper products. Their physical nature is different.
- Intended_purpose: Both ar

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

You are an expert in trademark law and NICE classification.
Your task is to evaluate the similarity between two products according to the official NICE classification and the 8 guideline factors: Nature, Intended Purpose, method of use, complementarity, competition, distribution channels, relevant public,the usual origin of the goods/services

Scoring system (for each factor and overall):
0 = Not similar, 1 = Slightly similar, 2 = Somewhat similar, 3 = Similar, 4 = Highly similar.

### Example USECASE_1
**Input:**
- Product 1: Make-up preparations (Class 3, Code 030033)
- Product 2: Tissues of paper for removing make-up (Class 16, Code 160294)
- Class Info: {
  "class_3": "Non-medicated toiletry preparations and cleaning products.",
  "class_16": "Mainly paper, cardboard and goods made from these materials."
}

**Reasoning:**
- Nature: Make-up preparations are chemical/beauty compositions, while tissues are paper products. Their physical nature is different.
- Intended_purpose: Both ar